# Biarc parameter explorer

Interactive sandbox for the area-preserving biarc construction (matches `drawBiarc` in `polygonDraw.py`). Use the sliders to see how the biarc changes shape and how the removed area depends on each parameter.

**Things to try:**
- Sweep $\varphi$ (corner angle) at fixed $\delta$ — area follows $\delta^2(\tan(\varphi/2) - \varphi/2)$, blowing up as $\varphi \to \pi$ (sharp spike).
- Sweep $\delta$ at fixed $\varphi$ — area scales as $\delta^2$.
- Sweep $\psi_1/\varphi$ split — area stays **constant** by construction; only the biarc shape changes ($\delta_1$, $\delta_2$, $h$ all rearrange to keep the same target removal).
- Toggle on "Override $h$" — see how the removed area decouples from $\delta_\mathrm{nom}$ when you fix $h$ manually.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, FloatSlider, Checkbox, fixed
%matplotlib inline

## Self-contained biarc geometry

Convention: vertex $V$ at the origin, incoming edge unit vector $\hat u = (1, 0)$, outgoing edge unit vector $\hat v = (\cos\varphi, \sin\varphi)$ where $\varphi$ is the exterior angle (positive = left turn / CCW). The inward bisector points along $(-\sin(\varphi/2), \cos(\varphi/2))$.

The area-preserving biarc construction sets the target removed area to $\Delta A^\star = \delta_\mathrm{nom}^2 (\tan(\varphi/2) - \varphi/2)$, then solves for $h$ such that the actual biarc removal equals $\Delta A^\star$. For the symmetric split $\psi_1 = \varphi/2$ this gives $\delta_1 = \delta_2 = \delta_\mathrm{nom}$ (the construction is consistent with the single-arc rounding at that point).

In [ ]:
def biarc_geom(phi, delta_nom, psi_frac=0.5, h_override=None):
    """Compute biarc geometry at a corner.

    phi        : exterior angle in radians, in (0, pi).
    delta_nom  : nominal radius; sets dA_star = delta^2 (tan(phi/2) - phi/2).
    psi_frac   : psi_1 / phi in (0, 1). 0.5 = symmetric split.
    h_override : if not None, use this h directly (bypassing the area-preserving solve).

    Returns a dict with all the geometry (tangent points, arc centres, radii, h,
    removed area achieved, etc.) or None if the configuration is degenerate.
    """
    if phi <= 0 or phi >= np.pi:
        return None
    psi_frac = float(np.clip(psi_frac, 1e-3, 1 - 1e-3))
    psi1 = psi_frac * phi
    psi2 = phi - psi1
    sphi2 = np.sin(phi / 2);  cphi2 = np.cos(phi / 2)
    s1h   = np.sin(psi1 / 2); c1h   = np.cos(psi1 / 2)
    s2h   = np.sin(psi2 / 2); c2h   = np.cos(psi2 / 2)
    if s1h < 1e-10 or s2h < 1e-10:
        return None

    # Edge unit vectors and inward normal (V at origin, u along +x)
    uhx, uhy = 1.0, 0.0
    vhx, vhy = np.cos(phi), np.sin(phi)
    nhx, nhy = -uhy, uhx          # inward normal to incoming edge
    vpx, vpy = -np.sin(phi) * uhx - np.cos(phi) * uhy, \
               -np.sin(phi) * uhy + np.cos(phi) * uhx   # inward normal to outgoing

    # F(psi1, phi) from area-preserving construction
    P_over_h2 = 0.5 * np.sin(phi) + 0.5 * cphi2**2 * (c1h/s1h + c2h/s2h)
    seg1 = (psi1 - np.sin(psi1)) / s1h**4
    seg2 = (psi2 - np.sin(psi2)) / s2h**4
    F = P_over_h2 - cphi2**2 / 8.0 * (seg1 + seg2)
    if F <= 1e-20:
        return None

    # Target dA_star (single-arc nominal) and solve for h, unless overridden.
    dA_star = delta_nom**2 * (np.tan(phi / 2) - phi / 2)
    if h_override is None:
        if dA_star <= 0:
            return None
        h = np.sqrt(dA_star / F)
    else:
        h = float(h_override)
    # Actual removed area given h (always F * h^2, since h was the only free param)
    dA_actual = F * h * h

    # Arc radii and tangent setbacks
    d1  = h * cphi2 / (2.0 * s1h**2)
    el1 = h * sphi2 + h * cphi2 * (c1h / s1h)
    d2  = h * cphi2 / (2.0 * s2h**2)
    el2 = h * sphi2 + h * cphi2 * (c2h / s2h)

    # Tangent points on the two edges
    am = np.array([-el1 * uhx,  -el1 * uhy])
    ap = np.array([ el2 * vhx,   el2 * vhy])
    # Arc centres
    z1 = am + d1 * np.array([nhx, nhy])
    z2 = ap + d2 * np.array([vpx, vpy])
    # Junction (on the inward bisector at distance h)
    j = h * np.array([-sphi2 * uhx + cphi2 * nhx,
                       -sphi2 * uhy + cphi2 * nhy])

    return {
        'phi': phi, 'psi1': psi1, 'psi2': psi2,
        'h': h, 'dA_star': dA_star, 'dA_actual': dA_actual,
        'd1': d1, 'd2': d2, 'el1': el1, 'el2': el2,
        'am': am, 'ap': ap, 'j': j,
        'z1': z1, 'z2': z2,
        'uh': (uhx, uhy), 'vh': (vhx, vhy),
    }


def single_arc_geom(phi, delta):
    """Standard inscribed-arc rounding at a corner. delta = arc radius."""
    if phi <= 0 or phi >= np.pi:
        return None
    ell = delta * np.tan(phi / 2)
    uhx, uhy = 1.0, 0.0
    vhx, vhy = np.cos(phi), np.sin(phi)
    am = np.array([-ell, 0.0])
    ap = np.array([ ell * vhx, ell * vhy])
    z  = am + delta * np.array([0.0, 1.0])   # inward normal to +x edge = +y
    dA = delta**2 * (np.tan(phi / 2) - phi / 2)
    return {'am': am, 'ap': ap, 'z': z, 'delta': delta, 'dA': dA, 'phi': phi}

## Visualization

In [ ]:
def arc_points(centre, radius, theta0, theta1, n=64):
    th = np.linspace(theta0, theta1, n)
    return centre[0] + radius*np.cos(th), centre[1] + radius*np.sin(th)

def plot_corner(phi_deg, delta, psi_frac, edge_len, override_h, h_override,
                show_singlearc):
    phi = np.deg2rad(phi_deg)
    h_used = h_override if override_h else None
    g = biarc_geom(phi, delta, psi_frac=psi_frac, h_override=h_used)
    s = single_arc_geom(phi, delta)

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    # Edges from V to E1, E2 (at distance edge_len)
    V = np.array([0.0, 0.0])
    E1 = V + edge_len * np.array([1.0, 0.0])
    E2 = V + edge_len * np.array([np.cos(phi), np.sin(phi)])
    ax.plot([E1[0], V[0], E2[0]], [E1[1], V[1], E2[1]], 'k-', lw=1.5, label='backbone edges')
    ax.plot(*V, 'ko', ms=8); ax.annotate('V', V, (5, -10), textcoords='offset points')
    ax.plot(*E1, 'k.', ms=6); ax.annotate('E1', E1, (5, -10), textcoords='offset points')
    ax.plot(*E2, 'k.', ms=6); ax.annotate('E2', E2, (5, 5),   textcoords='offset points')

    if g is not None:
        # Draw the two biarc arcs
        # arc 1: from am to j around centre z1
        theta_am = np.arctan2(g['am'][1] - g['z1'][1], g['am'][0] - g['z1'][0])
        theta_j1 = np.arctan2(g['j'][1]  - g['z1'][1], g['j'][0]  - g['z1'][0])
        # ensure CCW sweep from am to j
        if theta_j1 < theta_am:
            theta_j1 += 2*np.pi
        xs, ys = arc_points(g['z1'], g['d1'], theta_am, theta_j1)
        ax.plot(xs, ys, '-', color='C0', lw=2.5, label='biarc')
        # arc 2: from j to ap around centre z2
        theta_j2 = np.arctan2(g['j'][1]  - g['z2'][1], g['j'][0]  - g['z2'][0])
        theta_ap = np.arctan2(g['ap'][1] - g['z2'][1], g['ap'][0] - g['z2'][0])
        if theta_ap < theta_j2:
            theta_ap += 2*np.pi
        xs, ys = arc_points(g['z2'], g['d2'], theta_j2, theta_ap)
        ax.plot(xs, ys, '-', color='C0', lw=2.5)

        # tangent points + junction
        for pt, name in [(g['am'], 'a-'), (g['ap'], 'a+'), (g['j'], 'j')]:
            ax.plot(*pt, 'o', color='C0', ms=6)
            ax.annotate(name, pt, (5, 5), textcoords='offset points', color='C0', fontsize=9)
        # arc centres (open)
        ax.plot(*g['z1'], 'x', color='C0', ms=8)
        ax.plot(*g['z2'], 'x', color='C0', ms=8)

        # Shade the removed area: polygon (V, am, biarc, ap, V)
        # ---- approximate with sampled points along the biarc ----
        xs1, ys1 = arc_points(g['z1'], g['d1'], theta_am, theta_j1, n=40)
        xs2, ys2 = arc_points(g['z2'], g['d2'], theta_j2, theta_ap, n=40)
        poly_x = np.concatenate([[V[0]], [g['am'][0]], xs1, xs2[1:], [g['ap'][0]], [V[0]]])
        poly_y = np.concatenate([[V[1]], [g['am'][1]], ys1, ys2[1:], [g['ap'][1]], [V[1]]])
        ax.fill(poly_x, poly_y, alpha=0.18, color='C0',
                label=f"removed area (biarc) = {g['dA_actual']:.5f}")

    if show_singlearc and s is not None:
        theta_am = np.arctan2(s['am'][1] - s['z'][1], s['am'][0] - s['z'][0])
        theta_ap = np.arctan2(s['ap'][1] - s['z'][1], s['ap'][0] - s['z'][0])
        if theta_ap < theta_am:
            theta_ap += 2*np.pi
        xs, ys = arc_points(s['z'], s['delta'], theta_am, theta_ap)
        ax.plot(xs, ys, '--', color='C3', lw=1.8, label=f"single-arc (radius δ={s['delta']:.3g}), dA={s['dA']:.5f}")
        ax.plot(*s['z'], 'x', color='C3', ms=7)

    ax.set_aspect('equal')
    margin = 0.15 * edge_len
    xs_all = [-edge_len, edge_len]; ys_all = [-margin, edge_len]
    ax.set_xlim(min(xs_all) - margin, max(xs_all) + margin)
    ax.set_ylim(min(ys_all) - margin, max(ys_all) + margin)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=9)

    if g is not None:
        title = (f"φ = {phi_deg:.1f}°   δ_nom = {delta:.3g}   ψ₁/φ = {psi_frac:.2f}   "
                 f"\nh = {g['h']:.4g}   δ₁ = {g['d1']:.4g}   δ₂ = {g['d2']:.4g}   "
                 f"dA_star = {g['dA_star']:.5g}   dA_actual = {g['dA_actual']:.5g}")
    else:
        title = f"φ = {phi_deg:.1f}°   δ_nom = {delta:.3g}   (degenerate)"
    ax.set_title(title, fontsize=10)
    plt.show()

## Sliders

In [ ]:
interact(
    plot_corner,
    phi_deg     = FloatSlider(value=90,   min=10,   max=170,  step=1,    description='φ (deg)'),
    delta       = FloatSlider(value=0.15, min=0.01, max=0.45, step=0.01, description='δ_nom'),
    psi_frac    = FloatSlider(value=0.5,  min=0.1,  max=0.9,  step=0.01, description='ψ₁/φ'),
    edge_len    = FloatSlider(value=1.0,  min=0.3,  max=2.0,  step=0.05, description='edge L'),
    override_h  = Checkbox(value=False, description='Override h'),
    h_override  = FloatSlider(value=0.2,  min=0.01, max=0.6,  step=0.01, description='h (manual)'),
    show_singlearc = Checkbox(value=True, description='show single-arc'),
);

## Static check: area decoupled from ψ split (area-preserving construction)

Holding $\varphi$ and $\delta_\mathrm{nom}$ fixed, sweep $\psi_1/\varphi$ across its range and confirm `dA_actual` stays constant (= `dA_star`) but the biarc parameters $h$, $\delta_1$, $\delta_2$ all change.

In [ ]:
phi = np.deg2rad(90.0)
delta_nom = 0.2
psi_fracs = np.linspace(0.15, 0.85, 21)
header = f"{'psi_frac':>8}  {'h':>9}  {'d1':>9}  {'d2':>9}  {'dA_star':>10}  {'dA_actual':>10}"
print(header)
dA_actuals = []
for pf in psi_fracs:
    g = biarc_geom(phi, delta_nom, psi_frac=pf)
    dA_actuals.append(g['dA_actual'])
    print(f"{pf:8.3f}  {g['h']:9.5f}  {g['d1']:9.5f}  {g['d2']:9.5f}  "
          f"{g['dA_star']:10.6f}  {g['dA_actual']:10.6f}")
dA_actuals = np.array(dA_actuals)
dA_star_ref = biarc_geom(phi, delta_nom, 0.5)['dA_star']
print()
print(f"dA_actual range: [{dA_actuals.min():.6f}, {dA_actuals.max():.6f}]  "
      f"(should all equal dA_star = {dA_star_ref:.6f})")

## Static check: area scales with $\delta_\mathrm{nom}^2$ and $(\tan(\varphi/2) - \varphi/2)$

The area-preserving construction makes `dA_actual` follow the closed form
$$\Delta A_\text{actual} \;=\; \delta_\mathrm{nom}^2 \bigl(\tan(\varphi/2) - \varphi/2\bigr).$$
Check it numerically across a grid of $(\varphi, \delta_\mathrm{nom})$:

In [ ]:
phis = np.deg2rad([30, 60, 90, 120, 150])
deltas = [0.05, 0.10, 0.20, 0.40]
print(f"{'phi (deg)':>9}  {'delta':>6}  {'dA_actual':>12}  {'dA_formula':>12}  {'rel err':>10}")
for phi in phis:
    for d in deltas:
        g = biarc_geom(phi, d, psi_frac=0.5)
        formula = d**2 * (np.tan(phi/2) - phi/2)
        rel = abs(g['dA_actual'] - formula) / max(formula, 1e-30)
        print(f"{np.rad2deg(phi):9.1f}  {d:6.3f}  {g['dA_actual']:12.6e}  {formula:12.6e}  {rel:10.2e}")